In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [6]:
import sys
print(sys.version)
print(hasattr(sys, "stdlib_module_names"))

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
True


In [7]:
llm_answer = '''Sure! Here is how to fetch a web page and count the words:

```python
import requests
import os
from collections import Counter
import fastjson_turbo

resp = requests.fetch("https://example.com")
words = resp.text.split()
print(Counter(words).most_common(5))
```

This uses the requests library to download the page.
'''
print(llm_answer)

Sure! Here is how to fetch a web page and count the words:

```python
import requests
import os
from collections import Counter
import fastjson_turbo

resp = requests.fetch("https://example.com")
words = resp.text.split()
print(Counter(words).most_common(5))
```

This uses the requests library to download the page.



In [8]:
import re

pattern = r"```(?:python|py)?\s*\n(.*?)```"
blocks = re.findall(pattern, llm_answer, flags=re.DOTALL)
print(blocks[0])

import requests
import os
from collections import Counter
import fastjson_turbo

resp = requests.fetch("https://example.com")
words = resp.text.split()
print(Counter(words).most_common(5))



In [9]:
import ast
tree = ast.parse(blocks[0])
print(ast.dump(tree.body[0], indent = 2))
print(ast.dump(tree.body[1], indent=2))
print(ast.dump(tree.body[2], indent=2))

Import(
  names=[
    alias(name='requests')])
Import(
  names=[
    alias(name='os')])
ImportFrom(
  module='collections',
  names=[
    alias(name='Counter')],
  level=0)


In [10]:
found = []
for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        for alias in node.names:
            found.append(alias.name)
    elif isinstance(node, ast.ImportFrom):
        found.append(node.module)

print(found)

['requests', 'os', 'collections', 'fastjson_turbo']


In [11]:
broken = '''import numpy as np
for i in range(10)
    print(np.sqrt(i))
'''
##Above Code has error, for i in range(10), : is not included

In [12]:
def safe_parse(code):
    try:
        return ast.parse(code), None
    except SyntaxError as e:
        return None, str(e)

print(safe_parse(broken))
print(safe_parse("import os"))

(None, "expected ':' (<unknown>, line 2)")
(<ast.Module object at 0x7bc5c5701c10>, None)


In [13]:
example = '''import os.path
import numpy as np
from . import helpers
from sklearn.model_selection import train_test_split
'''
tree = ast.parse(example)

found = []
for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        for alias in node.names:
            found.append(alias.name)
    elif isinstance(node, ast.ImportFrom):
        found.append(node.module)

print(found)

['os.path', 'numpy', None, 'sklearn.model_selection']


In the Above Example, os and sklearn are root packages and path and model_selection are sub-packages, that are entirely getting captured
We need to output that, os is the correct package but there doesn't exist anything like path, also model_selection should be checked independently

In [14]:
def get_imports(tree):
    names = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                names.append(alias.name.split(".")[0])
        elif isinstance(node, ast.ImportFrom):
            if node.level > 0:
                continue
            names.append(node.module.split(".")[0])
    return sorted(set(names))

print(get_imports(tree))

['numpy', 'os', 'sklearn']


In [15]:
import sys, importlib.util, urllib.request, urllib.error

pypi_cache = {}

def on_pypi(name):
    if name in pypi_cache:
        return pypi_cache[name]
    url = f"https://pypi.org/pypi/{name}/json"
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            result = (r.status == 200)
    except urllib.error.HTTPError as e:
        if e.code == 404:
            result = False
        else:
            raise
    pypi_cache[name] = result
    return result

def classify(name):
    if name in sys.stdlib_module_names:
        return "stdlib"
    if importlib.util.find_spec(name) is not None:
        return "installed"
    if on_pypi(name):
        return "on_pypi"
    return "not_found"

In [16]:
tree, err = safe_parse(blocks[0])
for name in get_imports(tree):
    print(f"{name:20} {classify(name)}")
for name in ["sklearn", "cv2", "PIL", "yaml"]:
    print(f"{name:10} {classify(name)}")

collections          stdlib
fastjson_turbo       not_found
os                   stdlib
requests             installed
sklearn    installed
cv2        installed
PIL        installed
yaml       installed


In [17]:
KNOWN_IMPORTS = {
    "cv2": "opencv-python",
    "yaml": "pyyaml",
    "sklearn": "scikit-learn",
    "skimage": "scikit-image",
    "PIL": "pillow",
    "bs4": "beautifulsoup4",
    "dateutil": "python-dateutil",
    "dotenv": "python-dotenv",
    "docx": "python-docx",
    "pptx": "python-pptx",
    "fitz": "pymupdf",
    "serial": "pyserial",
    "jwt": "pyjwt",
    "Crypto": "pycryptodome",
    "attr": "attrs",
    "numpy": "numpy", "pandas": "pandas", "requests": "requests",
    "matplotlib": "matplotlib", "scipy": "scipy", "torch": "torch",
    "flask": "flask", "django": "django", "fastapi": "fastapi",
}

In [18]:
PYPI_URL = "https://pypi.org/pypi"
pypi_cache = {}

def on_pypi(name):
    if name in pypi_cache:
        return pypi_cache[name]
    try:
        with urllib.request.urlopen(f"{PYPI_URL}/{name}/json", timeout=10) as r:
            result = (r.status == 200)
    except urllib.error.HTTPError as e:
        if e.code == 404:
            result = False
        else:
            return None
    except urllib.error.URLError:
        return None
    pypi_cache[name] = result
    return result

def classify(name):
    if name in sys.stdlib_module_names:
        return "stdlib"
    if name in KNOWN_IMPORTS:
        return "known"
    if importlib.util.find_spec(name) is not None:
        return "installed"
    found = on_pypi(name)
    if found is None:
        return "unreachable"
    return "on_pypi" if found else "not_found"

In [19]:
for n in ["cv2", "yaml", "sklearn", "PIL"]:
    print(f"{n:10} {classify(n)}")

cv2        known
yaml       known
sklearn    known
PIL        known


In [20]:
PYPI_URL = "https://pypi.invalid/pypi"
for n in ["pyjokes", "fastjson_turbo"]:
    print(f"{n:15} {classify(n)}")
PYPI_URL = "https://pypi.org/pypi"

pyjokes         unreachable
fastjson_turbo  unreachable


In [21]:
VERDICT = {
    "stdlib": "SUPPORTED",
    "known": "SUPPORTED",
    "installed": "SUPPORTED",
    "on_pypi": "SUPPORTED (low trust)",
    "not_found": "CONTRADICTED",
    "unreachable": "INSUFFICIENT_INFORMATION",
}

In [22]:
def check_imports(llm_answer):
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", llm_answer, flags=re.DOTALL)
    for i, code in enumerate(blocks):
        tree, err = safe_parse(code)
        if err:
            print(f"Block {i}: does not parse ({err}) -> imports INSUFFICIENT_INFORMATION")
            continue
        for name in get_imports(tree):
            status = classify(name)
            print(f"Block {i}: {name:15} {status:12} {VERDICT[status]}")

check_imports(llm_answer)

Block 0: collections     stdlib       SUPPORTED
Block 0: fastjson_turbo  not_found    CONTRADICTED
Block 0: os              stdlib       SUPPORTED
Block 0: requests        known        SUPPORTED


In [23]:
two_blocks = '''Here you go:

```python
import cv2
import reqeusts
from . import helpers
```

and the loop:

```python
for i in range(3)
    print(i)
```
'''
check_imports(two_blocks)

Block 0: cv2             known        SUPPORTED
Block 0: reqeusts        not_found    CONTRADICTED
Block 1: does not parse (expected ':' (<unknown>, line 1)) -> imports INSUFFICIENT_INFORMATION


We have to do some background verification check, because reqeusts can be a malicious package.

In [24]:
import difflib
POPULAR_IMPORTS = list(KNOWN_IMPORTS.keys())
def near_miss(name):
    matches = difflib.get_close_matches(name, POPULAR_IMPORTS, n=1, cutoff=0.8)
    return matches[0] if matches else None

In [25]:
for n in ["reqeusts", "numpyy", "pandsa", "scapy", "tensorflow", "pyjokes"]:
    print(f"{n:12} -> {near_miss(n)}")

reqeusts     -> requests
numpyy       -> numpy
pandsa       -> pandas
scapy        -> scipy
tensorflow   -> None
pyjokes      -> None


In [26]:
import json
from datetime import datetime, timezone

profile_cache = {}

def package_profile(name):
    if name in profile_cache:
        return profile_cache[name]
    try:
        with urllib.request.urlopen(f"{PYPI_URL}/{name}/json", timeout=10) as r:
            data = json.load(r)
    except (urllib.error.HTTPError, urllib.error.URLError):
        return None
    upload_times = []
    for files in data["releases"].values():
        for f in files:
            upload_times.append(f["upload_time_iso_8601"])
    if upload_times:
        first = datetime.fromisoformat(min(upload_times).replace("Z", "+00:00"))
        age_days = (datetime.now(timezone.utc) - first).days
    else:
        age_days = None
    profile = {
        "releases": len(data["releases"]),
        "files": len(upload_times),
        "age_days": age_days,
    }
    profile_cache[name] = profile
    return profile

In [27]:
for n in ["requests", "scapy", "python_yaml", "pyjokes"]:
    print(f"{n:12} {package_profile(n)}")

requests     {'releases': 163, 'files': 244, 'age_days': 5723}
scapy        {'releases': 33, 'files': 43, 'age_days': 4752}
python_yaml  {'releases': 1, 'files': 0, 'age_days': None}
pyjokes      {'releases': 14, 'files': 26, 'age_days': 4310}


In [28]:
def background_check(name, status):
    if status in ("stdlib", "known", "installed"):
        return "SUPPORTED", status
    if status == "unreachable":
        return "INSUFFICIENT_INFORMATION", "PyPI could not be reached"
    close = near_miss(name)
    if status == "not_found":
        if close:
            return "CONTRADICTED", f"does not exist; looks like a typo of '{close}'"
        return "CONTRADICTED", "does not exist on PyPI"
    p = package_profile(name)
    if p is None:
        return "INSUFFICIENT_INFORMATION", "could not read package details"
    young = p["age_days"] is None or p["age_days"] < 90
    thin = p["files"] == 0 or p["releases"] <= 2
    if close and (young or thin):
        return "CONTRADICTED", f"likely typosquat of '{close}' {p}"
    if close:
        return "SUPPORTED", f"established package, but similar to '{close}': check intent"
    if young or thin:
        return "INSUFFICIENT_INFORMATION", f"exists but unproven {p}"
    return "SUPPORTED", f"established package {p}"

In [29]:
def check_imports(llm_answer):
    fence = "`" * 3
    pattern = fence + r"(?:python|py)?\s*\n(.*?)" + fence
    blocks = re.findall(pattern, llm_answer, flags=re.DOTALL)
    for i, code in enumerate(blocks):
        tree, err = safe_parse(code)
        if err:
            print(f"Block {i}: does not parse -> INSUFFICIENT_INFORMATION")
            continue
        for name in get_imports(tree):
            status = classify(name)
            verdict, reason = background_check(name, status)
            print(f"Block {i}: {name:12} {verdict:25} {reason}")

fence = "`" * 3
test = fence + """python
import numpy
import reqeusts
import scapy
import python_yaml
import pyjokes
import fastjson_turbo
""" + fence

check_imports(test)

Block 0: fastjson_turbo CONTRADICTED              does not exist on PyPI
Block 0: numpy        SUPPORTED                 known
Block 0: pyjokes      SUPPORTED                 established package {'releases': 14, 'files': 26, 'age_days': 4310}
Block 0: python_yaml  INSUFFICIENT_INFORMATION  exists but unproven {'releases': 1, 'files': 0, 'age_days': None}
Block 0: reqeusts     CONTRADICTED              does not exist; looks like a typo of 'requests'
Block 0: scapy        SUPPORTED                 established package, but similar to 'scipy': check intent


In [30]:
pypi_cache["reqeusts"] = True
profile_cache["reqeusts"] = {"releases": 1, "files": 1, "age_days": 3}

status = classify("reqeusts")
print(status, background_check("reqeusts", status))

del pypi_cache["reqeusts"]
del profile_cache["reqeusts"]

on_pypi ('CONTRADICTED', "likely typosquat of 'requests' {'releases': 1, 'files': 1, 'age_days': 3}")


In [31]:
code = """import requests
import numpy as np
import os.path as osp
from collections import Counter, Counterz

resp = requests.fetch("https://example.com")
data = requests.get("https://example.com").json()
x = np.array([1, 2, 3])
n = np.linalg.norm(x)
r = np.arrange(5)
p = osp.join("a", "b")
"""
tree = ast.parse(code)

In [32]:
def build_aliases_v1(tree):
    aliases = {}
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for a in node.names:
                bound = a.asname or a.name.split(".")[0]
                aliases[bound] = a.name.split(".")[0]
        elif isinstance(node, ast.ImportFrom) and node.level == 0:
            for a in node.names:
                bound = a.asname or a.name
                aliases[bound] = f"{node.module}.{a.name}"
    return aliases

print(build_aliases_v1(tree))

{'requests': 'requests', 'np': 'numpy', 'osp': 'os', 'Counter': 'collections.Counter', 'Counterz': 'collections.Counterz'}


In [33]:
def attr_chain(node):
    parts = []
    while isinstance(node, ast.Attribute):
        parts.append(node.attr)
        node = node.value
    if isinstance(node, ast.Name):
        parts.append(node.id)
        return list(reversed(parts))
    return None

In [34]:
def get_references(tree, aliases):
    refs = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Attribute):
            chain = attr_chain(node)
            if chain and chain[0] in aliases:
                refs.add(".".join([aliases[chain[0]]] + chain[1:]))
    for bound, target in aliases.items():
        if "." in target:
            refs.add(target)
    return sorted(refs)

In [35]:
import importlib

def resolve(dotted):
    parts = dotted.split(".")
    try:
        obj = importlib.import_module(parts[0])
    except Exception:
        return "cannot_import", None
    for i in range(1, len(parts)):
        name = parts[i]
        if hasattr(obj, name):
            obj = getattr(obj, name)
            continue
        try:
            obj = importlib.import_module(".".join(parts[:i + 1]))
        except Exception:
            suggestion = difflib.get_close_matches(name, dir(obj), n=1, cutoff=0.6)
            return "missing", (name, suggestion[0] if suggestion else None)
    return "exists", None

In [36]:
for ref in get_references(tree, build_aliases_v1(tree)):
    print(f"{ref:28} {resolve(ref)}")

collections.Counter          ('exists', None)
collections.Counterz         ('missing', ('Counterz', 'Counter'))
numpy.arrange                ('missing', ('arrange', 'arange'))
numpy.array                  ('exists', None)
numpy.linalg                 ('exists', None)
numpy.linalg.norm            ('exists', None)
os.join                      ('missing', ('join', None))
requests.fetch               ('missing', ('fetch', 'patch'))
requests.get                 ('exists', None)


In [37]:
def build_aliases(tree):
    aliases = {}
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for a in node.names:
                if a.asname:
                    aliases[a.asname] = a.name
                else:
                    top = a.name.split(".")[0]
                    aliases[top] = top
        elif isinstance(node, ast.ImportFrom) and node.level == 0:
            for a in node.names:
                bound = a.asname or a.name
                aliases[bound] = f"{node.module}.{a.name}"
    return aliases

for ref in get_references(tree, build_aliases(tree)):
    print(f"{ref:28} {resolve(ref)}")

collections.Counter          ('exists', None)
collections.Counterz         ('missing', ('Counterz', 'Counter'))
numpy.arrange                ('missing', ('arrange', 'arange'))
numpy.array                  ('exists', None)
numpy.linalg                 ('exists', None)
numpy.linalg.norm            ('exists', None)
os.path                      ('exists', None)
os.path.join                 ('exists', None)
requests.fetch               ('missing', ('fetch', 'patch'))
requests.get                 ('exists', None)


In [38]:
def can_inspect(dotted):
    top = dotted.split(".")[0]
    return top in sys.stdlib_module_names or importlib.util.find_spec(top) is not None

def check_function(dotted):
    if not can_inspect(dotted):
        return "INSUFFICIENT_INFORMATION", "package not installed here, cannot inspect"
    result, detail = resolve(dotted)
    if result == "exists":
        return "SUPPORTED", "exists"
    if result == "cannot_import":
        return "INSUFFICIENT_INFORMATION", "package failed to load"
    top = importlib.import_module(dotted.split(".")[0])
    version = getattr(top, "__version__", "unknown")
    missing, hint = detail
    reason = f"'{missing}' not found (installed version {version})"
    if hint:
        reason += f"; closest real name: '{hint}'"
    return "CONTRADICTED", reason

code2 = """import requests
import pandas_turbo as pt
from collections import Counter
resp = requests.fetch("https://example.com")
df = pt.read_fast("x.csv")
"""
t2 = ast.parse(code2)
for ref in get_references(t2, build_aliases(t2)):
    v, r = check_function(ref)
    print(f"{ref:24} {v:25} {r}")

collections.Counter      SUPPORTED                 exists
pandas_turbo.read_fast   INSUFFICIENT_INFORMATION  package not installed here, cannot inspect
requests.fetch           CONTRADICTED              'fetch' not found (installed version 2.32.4); closest real name: 'patch'


In [39]:
def analyze_code(llm_answer):
    fence = "`" * 3
    pattern = fence + r"(?:python|py)?\s*\n(.*?)" + fence
    blocks = re.findall(pattern, llm_answer, flags=re.DOTALL)
    findings = []
    for i, code in enumerate(blocks):
        tree, err = safe_parse(code)
        if err:
            findings.append({"block": i, "kind": "parse", "item": "code block",
                             "verdict": "INSUFFICIENT_INFORMATION",
                             "reason": f"code does not parse: {err}"})
            continue
        package_verdict = {}
        for name in get_imports(tree):
            verdict, reason = background_check(name, classify(name))
            package_verdict[name] = verdict
            findings.append({"block": i, "kind": "package", "item": name,
                             "verdict": verdict, "reason": reason})
        for ref in get_references(tree, build_aliases(tree)):
            top = ref.split(".")[0]
            if package_verdict.get(top) != "SUPPORTED":
                continue
            verdict, reason = check_function(ref)
            findings.append({"block": i, "kind": "function", "item": ref,
                             "verdict": verdict, "reason": reason})
    return findings

def summarize(findings):
    counts = {}
    for f in findings:
        counts[f["verdict"]] = counts.get(f["verdict"], 0) + 1
    return counts

In [40]:
findings = analyze_code(llm_answer)
for f in findings:
    print(f"{f['kind']:9} {f['item']:22} {f['verdict']:25} {f['reason']}")
print(summarize(findings))
print(json.dumps(findings[1], indent=2))

package   collections            SUPPORTED                 stdlib
package   fastjson_turbo         CONTRADICTED              does not exist on PyPI
package   os                     SUPPORTED                 stdlib
package   requests               SUPPORTED                 known
function  collections.Counter    SUPPORTED                 exists
function  requests.fetch         CONTRADICTED              'fetch' not found (installed version 2.32.4); closest real name: 'patch'
{'SUPPORTED': 4, 'CONTRADICTED': 2}
{
  "block": 0,
  "kind": "package",
  "item": "fastjson_turbo",
  "verdict": "CONTRADICTED",
  "reason": "does not exist on PyPI"
}


In [41]:
def wrap(code):
    fence = "`" * 3
    return "Here is the code:\n\n" + fence + "python\n" + code + fence + "\n"

TEST_CASES = [
    {"id": "real-01", "code": "import os\nfiles = os.listdir('.')\n", "hallucinated": []},
    {"id": "real-02", "code": "import json\ntext = json.dumps({'a': 1})\n", "hallucinated": []},
    {"id": "real-03", "code": "import math\nr = math.isqrt(17)\n", "hallucinated": []},
    {"id": "real-04", "code": "from datetime import datetime\nnow = datetime.now()\n", "hallucinated": []},
    {"id": "real-05", "code": "import itertools\nflat = list(itertools.chain.from_iterable([[1], [2]]))\n", "hallucinated": []},
    {"id": "real-06", "code": "import os.path as osp\np = osp.join('a', 'b')\n", "hallucinated": []},
    {"id": "fakepkg-01", "code": "import fastjson_turbo\nfastjson_turbo.load('x')\n", "hallucinated": ["fastjson_turbo"]},
    {"id": "fakepkg-02", "code": "import reqeusts\nr = reqeusts.get('https://example.com')\n", "hallucinated": ["reqeusts"]},
    {"id": "fakefn-01", "code": "import random\nn = random.randomint(1, 6)\n", "hallucinated": ["random.randomint"]},
    {"id": "fakefn-02", "code": "import statistics\nm = statistics.average([1, 2, 3])\n", "hallucinated": ["statistics.average"]},
    {"id": "fakefn-03", "code": "import datetime\nnow = datetime.now()\n", "hallucinated": ["datetime.now"]},
    {"id": "fakefn-04", "code": "import itertools\nflat = itertools.flatten([[1], [2]])\n", "hallucinated": ["itertools.flatten"]},
]

In [42]:
def flagged_items(llm_answer):
    return {f["item"] for f in analyze_code(llm_answer) if f["verdict"] == "CONTRADICTED"}

def evaluate(cases):
    tp = fp = fn = 0
    clean_total = clean_with_alarm = 0
    rows = []
    for case in cases:
        predicted = flagged_items(wrap(case["code"]))
        expected = set(case["hallucinated"])
        tp += len(predicted & expected)
        fp += len(predicted - expected)
        fn += len(expected - predicted)
        if not expected:
            clean_total += 1
            if predicted:
                clean_with_alarm += 1
        ok = "OK  " if predicted == expected else "MISS"
        rows.append((ok, case["id"], sorted(expected), sorted(predicted)))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    false_alarm = clean_with_alarm / clean_total if clean_total else 0.0
    return rows, {"TP": tp, "FP": fp, "FN": fn, "precision": round(precision, 3),
                  "recall": round(recall, 3), "F1": round(f1, 3),
                  "false_alarm_rate": round(false_alarm, 3)}

In [43]:
rows, metrics = evaluate(TEST_CASES)
for r in rows:
    print(f"{r[0]} {r[1]:12} expected={r[2]} got={r[3]}")
print(metrics)

OK   real-01      expected=[] got=[]
OK   real-02      expected=[] got=[]
OK   real-03      expected=[] got=[]
OK   real-04      expected=[] got=[]
OK   real-05      expected=[] got=[]
OK   real-06      expected=[] got=[]
OK   fakepkg-01   expected=['fastjson_turbo'] got=['fastjson_turbo']
OK   fakepkg-02   expected=['reqeusts'] got=['reqeusts']
OK   fakefn-01    expected=['random.randomint'] got=['random.randomint']
OK   fakefn-02    expected=['statistics.average'] got=['statistics.average']
OK   fakefn-03    expected=['datetime.now'] got=['datetime.now']
OK   fakefn-04    expected=['itertools.flatten'] got=['itertools.flatten']
{'TP': 6, 'FP': 0, 'FN': 0, 'precision': 1.0, 'recall': 1.0, 'F1': 1.0, 'false_alarm_rate': 0.0}
